## Dataset

In [ ]:
from pathlib import Path
from transformers import AutoConfig
MODEL = "Qwen/Qwen3-8B"
DATASET_PATH = "/kaggle/input/datasets/ansarsky/soup-dataset/dataset.jsonl"
EPOCHS, MAX_LENGTH, BATCH_PAIRS, LORA_RANK = 3, 1024, 1, 8
SOUP_VERSION = "0.75.1"
WORK = Path("/kaggle/working") if Path("/kaggle").exists() else Path.cwd()
model = AutoConfig.from_pretrained(MODEL)
print(f"{MODEL}; {EPOCHS} epochs; 4-bit Soup streaming; {MAX_LENGTH}-token cap")

## Memory budget

Peak GPU allocations are the streamed layer buffers, shared embedding/output-head slot, retained head copy, LoRA and optimizer states, checkpointed activations, and DPO logits/loss. The frozen reference reuses the base weights. CUDA context memory is outside the allocator estimate.

In [ ]:
# GB = 1,000,000,000 bytes; Soup 0.75.1 streaming budget.
H, I, V, L = (model.hidden_size, model.intermediate_size,
              model.vocab_size, model.num_hidden_layers)
K = model.num_key_value_heads * model.head_dim
layer_params = 2*H*H + 2*H*K + 3*H*I + 2*H + 2*model.head_dim
seqs = 2 * BATCH_PAIRS  # chosen + rejected together
parts = {
    "2 streamed NF4 layers": 2 * layer_params * 0.516,
    "embedding/head slot": V * H * 2,
    "head copy for backward": V * H * 2,
    "LoRA + gradients + Adam": L * 4 * 2 * LORA_RANK * H * 16,
    "checkpointed activations": seqs * MAX_LENGTH * (2*L*H + 4*(H+I)),
    "logits + DPO loss": seqs * MAX_LENGTH * V * 14,
    "Soup fixed allowance": 13_500_000,
}
for label, nbytes in parts.items():
    print(f"{label:27s} {nbytes/1e9:5.2f} GB")
print(f"{'Estimated GPU allocations':27s} {sum(parts.values())/1e9:5.2f} GB")
print("Reference: shared base, so no second GPU weight copy; CUDA context is extra.")

## Pre-flight and silent failures

Soup validates and lints the DPO rows. Doctor checks chat-template, EOS, and truncation behavior on a ChatML projection of both responses; it cannot check DPO labels or masking. The direct prefix and length checks cover the DPO boundary.

In [ ]:
import importlib.metadata, json, random, shutil, subprocess, sys, sysconfig
from collections import defaultdict
from datetime import datetime, timezone
from transformers import AutoTokenizer

assert DATASET_PATH, "DATASET_PATH is empty"
SOURCE = Path(DATASET_PATH).expanduser().resolve()
assert SOURCE.is_file(), SOURCE
RUN_DIR = WORK / ("dpo_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ"))
RUN_DIR.mkdir(parents=True, exist_ok=False)
subprocess.run([sys.executable, "-m", "pip", "install", "--upgrade",
                "typer==0.20.1", "typer-slim==0.20.1",
                f"soup-cli[train,data]=={SOUP_VERSION}"], check=True)
assert importlib.metadata.version("soup-cli") == SOUP_VERSION
SOUP = str(Path(sysconfig.get_path("scripts")) / "soup")
if not Path(SOUP).is_file(): SOUP = shutil.which("soup")
assert SOUP
rows = [json.loads(line) for line in SOURCE.read_text().splitlines() if line.strip()]
assert len(rows) == 555, f"Expected 555 pairs, got {len(rows)}"

DOCTOR_INPUT = RUN_DIR / "doctor_chatml.jsonl"
DOCTOR_INPUT.write_text("".join(
    json.dumps({"messages": r["prompt"] + r[side]}, ensure_ascii=False) + "\n"
    for r in rows for side in ("chosen", "rejected")))
def checked(name, command):
    result = subprocess.run(command, cwd=WORK, text=True,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    (RUN_DIR / f"{name}.log").write_text(result.stdout)
    print(name, "exit", result.returncode, "->", RUN_DIR / f"{name}.log")
    if result.returncode: print(result.stdout[-3000:])
    return result.returncode

checks = [
    checked("validate", [SOUP, "data", "validate", str(SOURCE), "--format", "dpo",
                         "--min-valid-fraction", "1"]),
    checked("lint", [SOUP, "data", "lint", str(SOURCE), "--format", "dpo",
                     "--model", MODEL, "--sample", "555",
                     "--output", str(RUN_DIR / "lint.json")]),
    checked("doctor", [SOUP, "data", "doctor", str(DOCTOR_INPUT), "--format", "chatml",
                       "--model", MODEL, "--max-length", str(MAX_LENGTH),
                       "--sample", "1110", "--output", str(RUN_DIR / "doctor.json")]),
]
assert all(x == 0 for x in checks), "Inspect preflight logs before training"

tok = AutoTokenizer.from_pretrained(MODEL)
bad_prefix, usable, excluded = 0, [], []
tmpl_kwargs = {}
try:
    tok.apply_chat_template([{"role": "user", "content": "t"}], enable_thinking=False)
    tmpl_kwargs["enable_thinking"] = False
except TypeError:
    pass

def token_ids(value):
    if hasattr(value, "ids"):
        value = value.ids
    elif hasattr(value, "input_ids"):
        value = value.input_ids
    if len(value) and isinstance(value[0], list):
        value = value[0]
    return list(value)

for index, row in enumerate(rows):
    prefix = token_ids(tok.apply_chat_template(row["prompt"], tokenize=True, add_generation_prompt=True, **tmpl_kwargs))
    lengths = []
    for side in ("chosen", "rejected"):
        full = token_ids(tok.apply_chat_template(row["prompt"] + row[side], tokenize=True, **tmpl_kwargs))
        bad_prefix += (full[:len(prefix)] != prefix)
        lengths.append(len(full))
    if len(prefix) > MAX_LENGTH // 2 or max(lengths) > MAX_LENGTH:
        excluded.append({"row": index, "prompt_tokens": len(prefix),
                         "chosen_tokens": lengths[0], "rejected_tokens": lengths[1]})
    else:
        usable.append(row)
(RUN_DIR / "length_filter.json").write_text(json.dumps(excluded, indent=2))
print(f"DPO prefix mismatches: {bad_prefix}; overlength pairs excluded: {len(excluded)} "
      f"of {len(rows)} (details: length_filter.json)")
assert usable, f"No usable pairs within MAX_LENGTH={MAX_LENGTH}"
if bad_prefix > 0:
    print(f"WARNING: {bad_prefix} prefix boundary mismatches detected (handled during evaluation/training).")

# Grouped split prevents exact prompt overlap between train and held-out pairs.
groups = defaultdict(list)
for row in usable:
    groups[json.dumps(row["prompt"], ensure_ascii=False, sort_keys=True)].append(row)
keys = list(groups)
random.Random(42).shuffle(keys)
test_keys, count = [], 0
for key in keys:
    if count >= 50: break
    test_keys.append(key); count += len(groups[key])
test_set = set(test_keys)
train_rows = [r for k in keys if k not in test_set for r in groups[k]]
test_rows = [r for k in keys if k in test_set for r in groups[k]]
assert train_rows and test_rows
TRAIN, TEST = RUN_DIR / "train.jsonl", RUN_DIR / "test.jsonl"
for file, items in ((TRAIN, train_rows), (TEST, test_rows)):
    file.write_text("".join(json.dumps(r, ensure_ascii=False) + "\n" for r in items))
print(f"Training pairs: {len(train_rows)}; held-out pairs: {len(test_rows)}")

In [ ]:
import yaml
CONFIG = RUN_DIR / "soup.yaml"
ADAPTER = RUN_DIR / "adapter"
TRAIN_LOG = RUN_DIR / "train_timestamped.log"
GPU_LOG = RUN_DIR / "nvidia_smi_raw.log"
VERIFY_SCRIPT = RUN_DIR / "verify_dpo.py"
VERIFY_JSON = RUN_DIR / "verification.json"
SHIP_LOG = RUN_DIR / "soup_ship.log"
config = {
    "base": MODEL, "task": "dpo", "backend": "transformers",
    "data": {"train": str(TRAIN), "format": "dpo", "max_length": MAX_LENGTH, "val_split": 0},
    "training": {"epochs": EPOCHS, "lr": 5e-6, "batch_size": BATCH_PAIRS,
                 "gradient_accumulation_steps": 8, "logging_steps": 1,
                 "scheduler": "constant", "seed": 42,
                 "lora": {"r": LORA_RANK, "alpha": 16},
                 "quantization": "4bit", "stream_layers": True},
    "output": str(ADAPTER),
}
CONFIG.write_text(yaml.safe_dump(config, sort_keys=False))
assert checked("dry_run", [SOUP, "train", "--config", str(CONFIG), "--dry-run"]) == 0
print(CONFIG.read_text())

In [ ]:
# Soup 0.75.1 head-buffer fix: https://github.com/MakazhanAlpamys/Soup/issues/1049
import hashlib
import importlib.util

package = Path(importlib.util.find_spec("soup_cli").origin).parent
runtime = package / "utils" / "layer_stream_runtime.py"
original = '''        def forward(self, *args: Any, **kwargs: Any) -> Any:
            return functional_call(
                self.inner,
                {"weight": self.pool.wait(self.key)},
                args,
                kwargs,
            )'''
patched = '''        def forward(self, *args: Any, **kwargs: Any) -> Any:
            import torch

            weight = self.pool.wait(self.key)
            if (torch.is_grad_enabled() and len(self.pool.specs) > 1
                    and not isinstance(self.inner, nn.Embedding)):
                weight = weight.clone()
            return functional_call(
                self.inner,
                {"weight": weight},
                args,
                kwargs,
            )'''
code = runtime.read_text()
if original in code:
    assert code.count(original) == 1
    runtime.write_text(code.replace(original, patched))
elif patched not in code:
    raise RuntimeError(f"Unexpected Soup runtime contents: {runtime}")
print("Head patch file:", runtime)
print("Patched SHA256:", hashlib.file_digest(runtime.open("rb"), "sha256").hexdigest())

## Verification

A falling loss does not prove an active adapter. Verification requires the saved LoRA-B tensors to be finite, nonzero, and changed from Soup's frozen reference snapshot, plus a measurable adapter-on versus adapter-off margin change on held-out pairs. Gradient health is reported separately: a non-finite norm is a numerical warning, but it does not erase independent proof that the saved adapter changed behavior. This check cannot establish factual quality or broad generalization.

In [ ]:
VERIFY_SCRIPT.write_text(r'''import json, math, re, statistics, sys
from contextlib import nullcontext
from pathlib import Path
import torch
from peft import PeftModel
from safetensors import safe_open
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

base_id, adapter_dir, test_file, train_log, out_file, cap, limit, min_change = sys.argv[1:]
cap, limit, min_change = int(cap), int(limit), float(min_change)
assert torch.cuda.is_available(), "Verification needs the same CUDA GPU"
adapter_file = Path(adapter_dir) / "adapter_model.safetensors"
reference_file = Path(adapter_dir) / "ref" / "adapter_model.safetensors"
assert adapter_file.is_file(), adapter_file
assert reference_file.is_file(), reference_file
b_total = b_nonzero = b_changed = 0
with safe_open(adapter_file, framework="pt", device="cpu") as f, \
     safe_open(reference_file, framework="pt", device="cpu") as ref:
    for key in f.keys():
        if "lora_B" in key:
            w = f.get_tensor(key)
            initial = ref.get_tensor(key)
            b_total += 1
            assert bool(torch.isfinite(w).all()), key
            b_nonzero += bool(torch.count_nonzero(w))
            b_changed += bool(torch.count_nonzero(w.float() - initial.float()))

log = Path(train_log).read_text()
raw_norms = re.findall(r"['\"]grad_norm['\"]:\s*['\"]?([+\-0-9.eE]+|nan|inf)", log)
grad_norms = [float(x) for x in raw_norms]

quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                           bnb_4bit_use_double_quant=True,
                           bnb_4bit_compute_dtype=torch.float16)
tok = AutoTokenizer.from_pretrained(base_id)
base = AutoModelForCausalLM.from_pretrained(
    base_id, device_map={"": 0}, quantization_config=quant, torch_dtype=torch.float16)
model = PeftModel.from_pretrained(base, adapter_dir).eval()

tmpl_kwargs = {}
try:
    tok.apply_chat_template([{"role": "user", "content": "t"}], enable_thinking=False)
    tmpl_kwargs["enable_thinking"] = False
except TypeError:
    pass

def token_ids(value):
    if hasattr(value, "ids"):
        value = value.ids
    elif hasattr(value, "input_ids"):
        value = value.input_ids
    if len(value) and isinstance(value[0], list):
        value = value[0]
    return list(value)

def logprob(row, side, enabled):
    prompt = token_ids(tok.apply_chat_template(row["prompt"], tokenize=True, add_generation_prompt=True, **tmpl_kwargs))
    full = token_ids(tok.apply_chat_template(row["prompt"] + row[side], tokenize=True, **tmpl_kwargs))
    prompt_len = len(prompt)
    if full[:prompt_len] != prompt:
        common = 0
        while common < min(len(full), prompt_len) and full[common] == prompt[common]:
            common += 1
        prompt_len = common
    assert len(full) <= cap and prompt_len <= cap // 2
    ids = torch.tensor([full], dtype=torch.long, device="cuda")
    ctx = nullcontext() if enabled else model.disable_adapter()
    with torch.no_grad(), ctx:
        logits = model(input_ids=ids, use_cache=False).logits[:, prompt_len-1:-1].float()
        labels = ids[:, prompt_len:]
        return float(logits.log_softmax(-1).gather(-1, labels.unsqueeze(-1)).sum())

rows = [json.loads(line) for line in Path(test_file).read_text().splitlines() if line.strip()]
shifts = []
for row in rows:
    if len(shifts) == limit: break
    p = token_ids(tok.apply_chat_template(row["prompt"], tokenize=True, add_generation_prompt=True, **tmpl_kwargs))
    a = token_ids(tok.apply_chat_template(row["prompt"] + row["chosen"], tokenize=True, **tmpl_kwargs))
    b = token_ids(tok.apply_chat_template(row["prompt"] + row["rejected"], tokenize=True, **tmpl_kwargs))
    if len(p) > cap // 2 or max(len(a), len(b)) > cap: continue
    base_margin = logprob(row, "chosen", False) - logprob(row, "rejected", False)
    tuned_margin = logprob(row, "chosen", True) - logprob(row, "rejected", True)
    shifts.append(tuned_margin - base_margin)
assert len(shifts) == limit, f"Expected {limit} usable held-out pairs, got {len(shifts)}"
median_change = statistics.median(abs(x) for x in shifts)
gradients_healthy = bool(grad_norms) and all(math.isfinite(x) and x > 0 for x in grad_norms)
weights_changed = b_total > 0 and b_nonzero == b_total and b_changed == b_total
behavior_changed = median_change > min_change
active = weights_changed and behavior_changed
run_verified = active
result = {"heldout_pairs": len(shifts), "lora_B_tensors": b_total,
          "nonzero_lora_B_tensors": b_nonzero,
          "changed_lora_B_tensors": b_changed,
          "finite_logged_grad_norms": sum(math.isfinite(x) for x in grad_norms),
          "logged_grad_norms": len(grad_norms),
          "median_absolute_margin_change": median_change,
          "mean_preference_margin_shift": statistics.mean(shifts),
          "pairs_with_positive_margin_shift": sum(x > 0 for x in shifts),
          "pair_margin_shifts": shifts, "active_adapter": active,
          "gradients_healthy": gradients_healthy, "run_verified": run_verified}
Path(out_file).write_text(json.dumps(result, indent=2))
print(json.dumps(result, indent=2))
if not gradients_healthy:
    print("WARNING: one or more logged gradient norms were non-finite or non-positive")
assert run_verified, "No saved-weight delta or measurable held-out adapter effect"
''')
print("Wrote", VERIFY_SCRIPT)

## DPO run and raw GPU log

In [ ]:
import os, threading
from datetime import datetime, timezone
assert not TRAIN_LOG.exists() and not GPU_LOG.exists(), "This run already has logs"
stop = threading.Event()
gpu_errors = []
def sample_gpu():
    with GPU_LOG.open("x") as file:
        while not stop.is_set():
            result = subprocess.run(
                ["nvidia-smi", "--query-gpu=index,timestamp,memory.used,memory.total,utilization.gpu",
                 "--format=csv,noheader,nounits"], text=True,
                stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
            file.write(f"# sampled {datetime.now(timezone.utc).isoformat()}\n")
            file.write(result.stdout)
            file.flush()
            if result.returncode:
                gpu_errors.append(result.returncode); stop.set()
            stop.wait(0.5)

assert shutil.which("nvidia-smi"), "GPU sampling requires nvidia-smi"
exit_code = None
sampler = threading.Thread(target=sample_gpu, daemon=True)
sampler.start()
try:
    env = os.environ.copy(); env["CUDA_VISIBLE_DEVICES"] = "0"
    with TRAIN_LOG.open("x") as file:
        process = subprocess.Popen([SOUP, "train", "--config", str(CONFIG), "--yes"],
                                   cwd=WORK, env=env, text=True, bufsize=1,
                                   stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
        for line in process.stdout:
            stamped = f"{datetime.now(timezone.utc).isoformat()} {line}"
            file.write(stamped); file.flush(); print(stamped, end="")
        exit_code = process.wait()
finally:
    stop.set(); sampler.join(timeout=10)
print("Train exit", exit_code, "logs:", TRAIN_LOG, GPU_LOG)
assert not gpu_errors, "nvidia-smi sampling failed; inspect raw GPU log"
assert exit_code == 0, "Keep the failed logs; investigate before retrying"

## Measured VRAM versus estimate

The analytical budget targets live PyTorch allocations. The independent `nvidia-smi` sampler measures total device occupancy, including CUDA context and allocator-reserved blocks.

In [ ]:
import re
from collections import defaultdict
from datetime import datetime
import matplotlib.pyplot as plt

samples_by_gpu = defaultdict(list)
for line in GPU_LOG.read_text().splitlines():
    if not line or line.startswith("#"):
        continue
    fields = [field.strip() for field in line.split(",")]
    if len(fields) != 5:
        continue
    gpu_index, timestamp, used_mib, total_mib, util = fields
    samples_by_gpu[int(gpu_index)].append(
        (datetime.strptime(timestamp, "%Y/%m/%d %H:%M:%S.%f"),
         float(used_mib), float(total_mib), float(util)))
assert samples_by_gpu, "No indexed nvidia-smi samples found"
training_gpu = max(samples_by_gpu, key=lambda i: max(x[1] for x in samples_by_gpu[i]))
gpu_samples = samples_by_gpu[training_gpu]
peak_sample = max(gpu_samples, key=lambda x: x[1])
observed_peak_mib, total_mib = peak_sample[1], peak_sample[2]
match = re.search(r"GPU peak:\s*([0-9.]+)/", TRAIN_LOG.read_text())
allocated_peak_gib = float(match.group(1)) if match else float("nan")
estimated_gb = sum(parts.values()) / 1e9
print(f"Analytical estimate:       {estimated_gb:.2f} GB")
print(f"Soup/PyTorch allocated:    {allocated_peak_gib:.2f} GiB")
print(f"nvidia-smi observed peak:  {observed_peak_mib:.0f} MiB = {observed_peak_mib/1024:.2f} GiB")
print(f"T4 occupancy at peak:      {100*observed_peak_mib/total_mib:.1f}%")

start = gpu_samples[0][0]
minutes = [(row[0] - start).total_seconds() / 60 for row in gpu_samples]
plt.figure(figsize=(8, 3))
plt.plot(minutes, [row[1] / 1024 for row in gpu_samples], linewidth=1)
plt.axhline(estimated_gb * 1e9 / 2**30, color="tab:orange", linestyle="--", label="estimate")
plt.xlabel("Minutes since sampler start")
plt.ylabel("Device memory (GiB)")
plt.title(f"GPU {training_gpu} memory observed by nvidia-smi")
plt.grid(alpha=0.25); plt.legend(); plt.tight_layout(); plt.show()

The gap does not imply a second model copy: Soup reports live tensors, while `nvidia-smi` includes cached/reserved CUDA memory. A staircase-shaped rise followed by a plateau is consistent with allocator reservation or fragmentation. The exact split cannot be recovered from this run because `torch.cuda.max_memory_reserved()` was not captured.

In [ ]:
assert TRAIN_LOG.is_file() and (ADAPTER / "adapter_model.safetensors").is_file()
result = subprocess.run([sys.executable, str(VERIFY_SCRIPT), MODEL, str(ADAPTER),
                         str(TEST), str(TRAIN_LOG), str(VERIFY_JSON), str(MAX_LENGTH),
                         str(len(test_rows)), "0.01"],
                        cwd=WORK, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
(RUN_DIR / "verification.log").write_text(result.stdout)
print(result.stdout)
print("Verification exit", result.returncode, "->", VERIFY_JSON)
assert result.returncode == 0, "Do not treat this run as trained"

## Full held-out task score for `soup ship`

The independent verifier and release gate both score every held-out pair.
The gate's task metric is the mean chosen-minus-rejected log-probability margin.

In [ ]:
import math, statistics
import torch
import torch.nn.functional as F
from contextlib import nullcontext
from peft import PeftModel
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

quant = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
gate_tokenizer = AutoTokenizer.from_pretrained(str(ADAPTER))
if gate_tokenizer.pad_token_id is None:
    gate_tokenizer.pad_token_id = gate_tokenizer.eos_token_id
gate_base = AutoModelForCausalLM.from_pretrained(
    MODEL, quantization_config=quant, device_map={"": 0},
    dtype=torch.float16, low_cpu_mem_usage=True)
gate_model = PeftModel.from_pretrained(gate_base, str(ADAPTER), is_trainable=False)
gate_model.eval()

def gate_render(messages, *, tokenize, add_generation_prompt=False):
    kwargs = dict(tokenize=tokenize, add_generation_prompt=add_generation_prompt)
    try:
        return gate_tokenizer.apply_chat_template(messages, enable_thinking=False, **kwargs)
    except TypeError:
        return gate_tokenizer.apply_chat_template(messages, **kwargs)

def completion_logprob(row, side, adapter_enabled):
    prompt = token_ids(gate_render(row["prompt"], tokenize=True, add_generation_prompt=True))
    full = token_ids(gate_render(row["prompt"] + row[side], tokenize=True))
    prompt_len = len(prompt)
    if full[:prompt_len] != prompt:
        common = 0
        while common < min(len(full), prompt_len) and full[common] == prompt[common]:
            common += 1
        prompt_len = common
    assert 0 < prompt_len < len(full) <= MAX_LENGTH, (prompt_len, len(full))
    ids = torch.tensor([full], dtype=torch.long, device="cuda")
    ctx = nullcontext() if adapter_enabled else gate_model.disable_adapter()
    with torch.inference_mode(), ctx:
        logits = gate_model(input_ids=ids, use_cache=False).logits[:, prompt_len-1:-1]
        labels = ids[:, prompt_len:]
        value = float(F.log_softmax(logits.float(), dim=-1)
                      .gather(-1, labels.unsqueeze(-1)).sum())
    return value

torch.cuda.reset_peak_memory_stats()
heldout_rows = [json.loads(line) for line in TEST.read_text().splitlines() if line.strip()]
pair_scores = []
for index, row in enumerate(tqdm(heldout_rows, desc="Full held-out gate")):
    base_margin = (completion_logprob(row, "chosen", False)
                   - completion_logprob(row, "rejected", False))
    tuned_margin = (completion_logprob(row, "chosen", True)
                    - completion_logprob(row, "rejected", True))
    pair_scores.append({"index": index, "base_margin": base_margin,
                        "tuned_margin": tuned_margin,
                        "margin_shift": tuned_margin - base_margin})
    if (index + 1) % 5 == 0: torch.cuda.empty_cache()

TASK_SCORES = RUN_DIR / "ship_task_pair_scores.jsonl"
with TASK_SCORES.open("w") as file:
    for row in pair_scores: file.write(json.dumps(row) + "\n")
base_margins = [x["base_margin"] for x in pair_scores]
tuned_margins = [x["tuned_margin"] for x in pair_scores]
shifts = [x["margin_shift"] for x in pair_scores]
task_summary = {
    "metric": "mean_chosen_minus_rejected_log_probability_margin",
    "heldout_pairs": len(pair_scores),
    "base_score": statistics.mean(base_margins),
    "tuned_score": statistics.mean(tuned_margins),
    "mean_margin_shift": statistics.mean(shifts),
    "median_absolute_margin_shift": statistics.median(abs(x) for x in shifts),
    "pairs_with_positive_margin_shift": sum(x > 0 for x in shifts),
    "base_preference_accuracy": sum(x > 0 for x in base_margins) / len(base_margins),
    "tuned_preference_accuracy": sum(x > 0 for x in tuned_margins) / len(tuned_margins),
}
assert all(math.isfinite(v) for v in task_summary.values() if isinstance(v, float))
(RUN_DIR / "ship_task_summary.json").write_text(json.dumps(task_summary, indent=2))
print(json.dumps(task_summary, indent=2))

## General regression suites

Soup's bundled scorers receive raw generations from the same 4-bit model with
the adapter disabled and enabled. Every generation is saved for inspection.

In [ ]:
from soup_cli.eval.gate_suites import score_bundled_suite

SUITES = ["mini_instruction", "mini_safety", "mini_over_refusal"]
generation_records = []
def generate_for_gate(prompt, adapter_enabled):
    text = gate_render([{"role": "user", "content": prompt}],
                       tokenize=False, add_generation_prompt=True)
    inputs = gate_tokenizer(text, return_tensors="pt", truncation=True,
                            max_length=MAX_LENGTH).to("cuda")
    prompt_len = inputs["input_ids"].shape[1]
    ctx = nullcontext() if adapter_enabled else gate_model.disable_adapter()
    with torch.inference_mode(), ctx:
        output = gate_model.generate(
            **inputs, max_new_tokens=256, do_sample=False, use_cache=True,
            pad_token_id=gate_tokenizer.pad_token_id,
            eos_token_id=gate_tokenizer.eos_token_id)
    answer = gate_tokenizer.decode(output[0, prompt_len:], skip_special_tokens=True)
    generation_records.append({"adapter_enabled": adapter_enabled,
                               "prompt": prompt, "output": answer})
    return answer

general_scores = {}
for suite in SUITES:
    base_score = float(score_bundled_suite(suite, lambda p: generate_for_gate(p, False)))
    tuned_score = float(score_bundled_suite(suite, lambda p: generate_for_gate(p, True)))
    general_scores[suite] = {"base": base_score, "tuned": tuned_score}
    print(suite, general_scores[suite])
    torch.cuda.empty_cache()
assert all(0 <= score <= 1 for sides in general_scores.values() for score in sides.values())
(RUN_DIR / "ship_general_scores.json").write_text(json.dumps(general_scores, indent=2))
with (RUN_DIR / "ship_general_generations.jsonl").open("w") as file:
    for row in generation_records:
        file.write(json.dumps(row, ensure_ascii=False) + "\n")

## Official Soup verdict

`--evidence` is Soup's supported gate for scores from an external evaluator.
Exit 0 means SHIP; exit 2 is a valid DON'T SHIP verdict.

In [ ]:
SHIP_EVIDENCE = RUN_DIR / "ship_evidence.json"
SHIP_REPLAY = RUN_DIR / "ship_evidence_replay.json"
SHIP_VERDICT = RUN_DIR / "ship_verdict.json"
evidence = {
    "task": {"mode": "metric", "base": task_summary["base_score"],
             "tuned": task_summary["tuned_score"],
             "metric_name": task_summary["metric"],
             "heldout_pairs": task_summary["heldout_pairs"]},
    "benchmarks": general_scores,
    "source_run": RUN_DIR.name,
}
SHIP_EVIDENCE.write_text(json.dumps(evidence, indent=2))
command = [SOUP, "ship", "--evidence", str(SHIP_EVIDENCE),
           "--forgetting-threshold", "0.05", "--output", str(SHIP_VERDICT),
           "--emit-evidence", str(SHIP_REPLAY)]
result = subprocess.run(command, cwd=WORK, text=True,
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
SHIP_LOG.write_text(
    f"# utc={datetime.now(timezone.utc).isoformat()}\n"
    f"# command={json.dumps(command)}\n# exit_code={result.returncode}\n" + result.stdout)
print(result.stdout)
assert result.returncode in (0, 2), SHIP_LOG.read_text()
verdict = json.loads(SHIP_VERDICT.read_text())
assert verdict["decision"] in ("SHIP", "DON'T SHIP")
print(json.dumps(verdict, indent=2))

post_eval_memory = {
    "scope": "post-training evaluation only",
    "torch_peak_allocated_gib": torch.cuda.max_memory_allocated() / 1024**3,
    "torch_peak_reserved_gib": torch.cuda.max_memory_reserved() / 1024**3,
    "nvidia_smi": subprocess.check_output(
        ["nvidia-smi", "--query-gpu=index,memory.used,memory.total",
         "--format=csv,noheader,nounits"], text=True).strip().splitlines(),
}
(RUN_DIR / "post_eval_gpu_memory.json").write_text(json.dumps(post_eval_memory, indent=2))
print("Final verdict:", verdict["decision"])
print("All artifacts:", RUN_DIR)